There, we'll:

Train Linear Regression
Train Decision Tree Regressor
Train Random Forest Regressor
Compare all models using MAE, RMSE, and R² Score
Save the best model for deployment

This is the stage where your crop yield predictor actually learns from the data.

# Model Training

## Objectives

- Load the preprocessed dataset.
- Split the data into training and testing sets.
- Train multiple regression models.
- Compare model performance.
- Select the best model.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
df = pd.read_csv("../data/processed/crop_yield_processed.csv")

df.head()

,Crop,Crop_Year,Season,State,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield
0,0,1997,4,2,73814.0,56708,2051.4,7024878.38,22882.34,0.796087
1,1,1997,1,2,6637.0,4685,2051.4,631643.29,2057.47,0.710435
2,8,1997,1,2,796.0,22,2051.4,75755.32,246.76,0.238333
3,9,1997,4,2,19656.0,126905000,2051.4,1870661.52,6093.36,5238.051739
4,11,1997,1,2,1739.0,794,2051.4,165500.63,539.09,0.420909


In [3]:
X = df.drop("Yield", axis=1)
y = df["Yield"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (19689, 9)
Target Shape: (19689,)


19689 rows → You have 19,689 crop records.
9 features → The model will use these 9 columns to predict yield:
Crop
Crop_Year
Season
State
Area
Production
Annual_Rainfall
Fertilizer
Pesticide
Target (y) → Yield

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples :", X_train.shape)
print("Testing samples  :", X_test.shape)

Training samples : (15751, 9)
Testing samples  : (3938, 9)


In [5]:
linear_model = LinearRegression()

linear_model.fit(X_train, y_train)

print("Linear Regression trained successfully!")

Linear Regression trained successfully!


In [6]:
y_pred_lr = linear_model.predict(X_test)

print(y_pred_lr[:10])

[53.11968005 53.90304706 57.67512399 52.75245502 52.75189412 53.2535221
 54.47998514 53.48957416 58.39699835 53.96109688]


In [7]:
mae = mean_absolute_error(y_test, y_pred_lr)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2 = r2_score(y_test, y_pred_lr)

print("Linear Regression Performance")
print("-----------------------------")
print("MAE :", mae)
print("RMSE:", rmse)
print("R² Score:", r2)

Linear Regression Performance
-----------------------------
MAE : 101.25373281405359
RMSE: 698.2419392244078
R² Score: 0.3915157906581028


Original Dataset
        │
        ▼
Features (X)        Target (y)
        │               │
        └──────┬────────┘
               ▼
        train_test_split()
               │
     ┌─────────┴─────────┐
     ▼                   ▼
X_train, y_train     X_test, y_test
     │
     ▼
linear_model.fit(X_train, y_train)
     │
     ▼
Model learns patterns
     │
     ▼
linear_model.predict(X_test)
     │
     ▼
Predicted Yield
     │
     ▼
Compare with y_test
     │
     ▼
MAE, RMSE, R²

Decision tree 

In [8]:
from sklearn.tree import DecisionTreeRegressor

dt_model = DecisionTreeRegressor(random_state=42)

dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)

print("Decision Tree trained successfully!")

Decision Tree trained successfully!


In [9]:
mae_dt = mean_absolute_error(y_test, y_pred_dt)
rmse_dt = np.sqrt(mean_squared_error(y_test, y_pred_dt))
r2_dt = r2_score(y_test, y_pred_dt)

print("Decision Tree Performance")
print("-------------------------")
print("MAE :", mae_dt)
print("RMSE:", rmse_dt)
print("R² Score:", r2_dt)

Decision Tree Performance
-------------------------
MAE : 11.784107148804976
RMSE: 324.6637761755164
R² Score: 0.8684456775342484


Random forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=30,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [11]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest Performance")
print("-------------------------")
print("MAE :", mae_rf)
print("RMSE:", rmse_rf)
print("R² Score:", r2_rf)

Random Forest Performance
-------------------------
MAE : 10.630760942785754
RMSE: 274.16664589527994
R² Score: 0.9061862083833261


Production is one of your input features.

Remember the relationship:

Yield=
Area
Production
	​
So we drop production as data leakage is happening

Version 1 (Current)
Includes Production
R² = 0.906
Shows excellent model performance.
Version 2 (Real-world)
Drops Production
More realistic because farmers won't know production before harvest.
This is the version you should deploy in the Gradio app.

This gives you a stronger project because you can discuss the trade-off in interviews.

Let's create Version 2

In [12]:
X = df.drop(["Yield", "Production"], axis=1)
y = df["Yield"]

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [15]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"m

In [16]:
y_pred_rf = rf_model.predict(X_test)

In [17]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest (Without Production)")
print("-------------------------------")
print("MAE :", mae_rf)
print("RMSE:", rmse_rf)
print("R² Score:", r2_rf)

Random Forest (Without Production)
-------------------------------
MAE : 10.260521568080213
RMSE: 131.39325516469623
R² Score: 0.9784531595856711


Let's inspect feature importance

In [18]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance)

           Feature  Importance
0             Crop    0.846236
3            State    0.097240
7        Pesticide    0.015015
6       Fertilizer    0.012902
4             Area    0.010737
1        Crop_Year    0.010049
5  Annual_Rainfall    0.007787
2           Season    0.000035


One thing catches my attention 🚨

An importance of 84.6% for Crop is very high.

That tells me:

Different crops have very different average yields.
The model relies heavily on crop identity.

This isn't necessarily wrong, but it's worth thinking about.

Let's make sure there isn't another issue

I want you to check one thing.

Run:

df.groupby("Crop")["Yield"].mean().sort_values(ascending=False)

Since Crop is encoded, you'll get numbers. If you still have the original CSV available, you can instead do:

In [19]:
df_original = pd.read_csv("../data/raw/crop_yield.csv")

df_original.groupby("Crop")["Yield"].mean().sort_values(ascending=False)

Crop
Coconut                  8652.000199
Sugarcane                  51.727439
Banana                     26.851128
Tapioca                    16.667301
Potato                     13.331718
Onion                      13.247525
Sweet potato                9.240788
Jute                        7.555393
Ginger                      6.442202
Mesta                       5.389204
Garlic                      4.544886
Maize                       3.427216
Turmeric                    3.325392
Cashewnut                   3.120438
Bajra                       2.427462
Rice                        2.218495
Tobacco                     2.110708
Dry chillies                2.078330
Arecanut                    2.073635
Wheat                       2.005086
Oilseeds total              1.995559
Cotton(lint)                1.797044
other oilseeds              1.789220
Barley                      1.595540
Peas & beans (Pulses)       1.393895
Groundnut                   1.360983
Sannhamp                    1.274